# Capstone — A Reproducible Spatial Data Science Workflow

**Goal:** integrate data loading, identity validation, descriptive analytics, quantile screening, cartographic display, and GeoJSON export.

**Research question:** Which synthetic teaching-grid cells have relatively high June land-surface temperatures? This example is a screening tool only; it does not estimate human exposure or causation.

## 1. Environment and teaching data

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Data audit, descriptive threshold, and quick visualization

We define the top-quartile threshold and **do not** claim that the threshold is an official risk standard. The output is a transparent class rule, not a machine-learning model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

cells = pd.read_csv(DATA / 'cells.csv')
obs = pd.read_csv(DATA / 'observations_clean.csv')
obs = obs.drop_duplicates(subset=['cell_id','date'])
june = cells.merge(obs.loc[obs.date == '2025-06-01',['cell_id','lst_c']],
                   on='cell_id',validate='one_to_one')
assert len(june) == 144 and june.lst_c.notna().all()
threshold = june.lst_c.quantile(0.75)
june['priority_flag'] = (june.lst_c >= threshold).astype(int)
print('Priority threshold (°C):', round(threshold,2))
print('Flagged cells:', int(june.priority_flag.sum()))
print(june.groupby('sector').agg(n_cells=('cell_id','size'),
                                mean_lst=('lst_c','mean'),
                                flagged=('priority_flag','sum')).round(2))
fig,ax = plt.subplots(figsize=(8,5.5))
sc = ax.scatter(june.easting_m,june.northing_m,c=june.lst_c,
                marker='s',s=220,cmap='coolwarm')
plt.colorbar(sc, ax=ax, label='Synthetic LST (°C)')
ax.set(aspect='equal',title='Synthetic June scene — temperature screening',
       xlabel='Easting (m)',ylabel='Northing (m)')
plt.show()


## 3. Export a GIS layer and optional HTML map

The join is keyed by cell ID; map colours show the chosen review flag. This is not itself a digital twin.

In [ ]:
geo = json.loads((DATA / 'cells.geojson').read_text(encoding='utf-8'))
look = june.set_index('cell_id')[['lst_c','priority_flag']].to_dict('index')
assert len(geo['features']) == len(june)
for feat in geo['features']:
    cid = feat['properties']['cell_id']
    feat['properties']['lst_c'] = round(float(look[cid]['lst_c']), 4)
    feat['properties']['priority_flag'] = int(look[cid]['priority_flag'])
output = Path('capstone_priority.geojson')
output.write_text(json.dumps(geo),encoding='utf-8')
print('Saved:', output.resolve())
print('GeoJSON feature count:', len(geo['features']))
try:
    import folium
    m = folium.Map(location=[3.12,101.68],zoom_start=12,tiles='CartoDB positron')
    folium.GeoJson(geo,style_function=lambda f:{'color':'#c2410c' if f['properties']['priority_flag'] else '#2a6072',
                                             'weight':1,'fillOpacity':0.42}).add_to(m)
    m.save('capstone_webgis.html')
    print('Saved capstone_webgis.html')
except ImportError:
    print('Optional interactive map skipped. Install folium using %pip install folium if needed.')


## 4. Student research assignment

1. Recalculate priorities for May and June and explain the difference.
2. Replace the threshold rule with a documented multi-criteria index.
3. Validate every geometry/key and record the CRS and date.
4. Evaluate sensitivity to cell size and the threshold.
5. Design a field verification plan and discuss limitations before operational use.